# The DAGScheduler: Converting DAGs into Execution Stages

**Module**: 04_execution_engine  
**Notebook**: 03_dag_scheduler.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How the DAGScheduler decomposes physical plans into stages, handles ShuffleMapStages vs ResultStages, and manages stage retries.
- **Why does this topic exist?**: The DAGScheduler is the brain that manages distributed execution pipelines.
- **What problem does it solve?**: Understanding fetch failures and stage re-computations.
- **Where is it used in real-world Data Engineering?**: Diagnosing pipeline failures in production.

---

## 2. DAGScheduler Architecture

The `DAGScheduler` works backward from the final action target, identifying `ShuffleDependencies` to draw stage boundaries. Stages with no dependencies run first.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("DAGScheduler_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: ShuffleMapStage vs ResultStage

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
res = orders.groupBy("status").count()
print("Trigger action to observe stages in Spark UI:")
res.show()
print("Visit http://localhost:4040/stages/ to see ShuffleMapStage (Stage 0) and ResultStage (Stage 1)!")

---

## Experiment: Stage Retries on Fetch Failure

- **1. What are we testing?**: Explain DAGScheduler retry policy.
- **2. What do we expect?**: Upstream stage is retried if shuffle files are lost.

In [ ]:
print('Max stage failures:', spark.conf.get('spark.stage.maxConsecutiveAttempts', '4'))

**3. What happened?**: Default 4 attempts before aborting application.  
**4. Why did it happen?**: Transient network drops can cause fetch failures.  
**5. Practical Engineering Lesson**: DAGScheduler retries stages automatically upon transient fetch errors.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Stages
**Problem Statement**: Count stages in a simple select query.  
**Input / Context**: orders.select('order_id')

In [ ]:
# Sol 1

### Exercise 2 (Intermediate): Two Shuffles Stage Count
**Problem Statement**: Count stages in groupBy + repartition.  
**Input / Context**: Provided dataset

In [ ]:
# Sol 2

### Exercise 3 (Advanced): DAGScheduler Metrics
**Problem Statement**: Check stage duration in UI.  
**Input / Context**: Provided dataset

In [ ]:
# Sol 3

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Stages):
```python
orders.select('order_id').count() # 1 stage
```

#### Solution 2 (Two Shuffles Stage Count):
```python
orders.groupBy('status').count().repartition(4).count() # 3 stages
```

#### Solution 3 (DAGScheduler Metrics):
```python
print('Inspected UI Stages tab')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **DAGScheduler**: Splits DAG at shuffles into stages.

### Essential Technical Terms
- `ShuffleMapStage`: Writes shuffle files.
- `ResultStage`: Computes final action.

### Common Engineering Mistakes
- **Assuming 1 job = 1 stage**: Jobs have multiple stages if shuffles exist.

### Interview Questions on This Topic
- **Q: What creates a stage boundary?**
  - *A*: A wide dependency (ShuffleDependency).

### What to Study Next
Open 04_task_scheduler.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')